In [4]:
# generate dummy ipout data: array of 500, 500, 84 

# genrate dumy ground truth: 500, 500 



# install cupy based on cuda version /
# pip install cupy-cuda11x

In [1]:
# check easting and northing of data 
import sys 
import os 
sys.path.append('/home/grace.bcolverd/2024-ESL-BiomassSAR')

from src.fdlbiomass.tomograph_reconstruction.utils import load_igs_data, read_SLC

import matplotlib.pyplot as plt 
import numpy as np 


In [6]:

heading='NW'
polarimetric_channel='HH'
parent_path = '/home/grace.bcolverd/esl_dev/tomosar-matlab/P'
output_path='/home/grace.bcolverd/local_data'


sar_data_dict = read_SLC(parent_path, heading, polarimetric_channel)

heading='SE'
se_sar_data_dict = read_SLC(parent_path, heading, polarimetric_channel)

nw_i, se_i = sar_data_dict['I_SLC'], se_sar_data_dict['I_SLC']
# igs = load_igs_data(parent_path, heading, polarimetric_channel, output_path)

Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722071511_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...
done.
Loading /home/grace.bcolverd/esl_dev/tomosar-matlab/P/NW/Util/SLC/kz/kz_20200722072647_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc.dat...


KeyboardInterrupt: 

In [ ]:

# Size: CrossRange: 23904, GroundRange: 1400
# Cross Range (azimuth_axis) Min: -6518.5, Max: 5433
# Ground Range (ground_range_axis) Min: 300, Max: 4497

In [2]:
nw_i.shape

(1274, 12953, 28)

In [3]:
dummy_data = np.random.rand(*(100, 1250, 84 ))

dummy_data.shape

(100, 1200, 84)

In [14]:
se_sar_data_dict['ground_range_axis']

array([ 510.,  513.,  516., ..., 4419., 4422., 4425.], dtype=float32)

In [95]:
se_sar_data_dict['I_SLC'].shape

(1306, 13530, 28)

In [7]:
sar_data_dict['ground_range_axis'],  se_sar_data_dict['ground_range_axis']

(array([ 606.,  609.,  612., ..., 4419., 4422., 4425.], dtype=float32),
 array([ 510.,  513.,  516., ..., 4419., 4422., 4425.], dtype=float32))

In [6]:
sar_data_dict['azimuth_axis'], se_sar_data_dict['azimuth_axis']


(array([-3400.5, -3400. , -3399.5, ...,  3074.5,  3075. ,  3075.5],
       dtype=float32),
 array([-3646. , -3645.5, -3645. , ...,  3117.5,  3118. ,  3118.5],
       dtype=float32))

In [8]:
sar_data, se_sar_data = sar_data_dict['I_SLC'], se_sar_data_dict['I_SLC']

In [1]:
sar_data.shape

NameError: name 'sar_data' is not defined

In [5]:
nw_i.shape , se_i.shape

((1274, 12953, 28), (1306, 13530, 28))

In [17]:
sar_data_dict.keys()

dict_keys(['scene_parameters', 'ground_range_axis', 'azimuth_axis', 'kz', 'focusing_centers', 'meta_sensing_dtm', 'tomo_dtm_sr', 'pol_ind', 'I_SLC'])

In [10]:
import numpy as np
from scipy import interpolate

# Your original axes
sar_azimuth = sar_data_dict['azimuth_axis']
se_sar_azimuth = se_sar_data_dict['azimuth_axis']
sar_ground_range = sar_data_dict['ground_range_axis']
se_sar_ground_range = se_sar_data_dict['ground_range_axis'] 

# Assuming sar_data and se_sar_data are your 3D arrays

# Find the full range for both axes
min_azimuth = min(sar_azimuth.min(), se_sar_azimuth.min())
max_azimuth = max(sar_azimuth.max(), se_sar_azimuth.max())
min_ground_range = min(sar_ground_range.min(), se_sar_ground_range.min())
max_ground_range = max(sar_ground_range.max(), se_sar_ground_range.max())

# Create new axes with the same step sizes
azimuth_step = sar_azimuth[1] - sar_azimuth[0]
ground_range_step = sar_ground_range[1] - sar_ground_range[0]
new_azimuth = np.arange(min_azimuth, max_azimuth + azimuth_step, azimuth_step)
new_ground_range = np.arange(min_ground_range, max_ground_range + ground_range_step, ground_range_step)

# Create meshgrids for interpolation
new_gr_mesh, new_az_mesh = np.meshgrid(new_ground_range, new_azimuth)

# Create 3D interpolation functions for both datasets
sar_interp = interpolate.RegularGridInterpolator((sar_ground_range, sar_azimuth, np.arange(sar_data.shape[2])), 
                                                 sar_data, bounds_error=False, fill_value=np.nan)
se_sar_interp = interpolate.RegularGridInterpolator((se_sar_ground_range, se_sar_azimuth, np.arange(se_sar_data.shape[2])), 
                                                    se_sar_data, bounds_error=False, fill_value=np.nan)

# Prepare coordinates for interpolation
coords = np.stack(np.meshgrid(new_ground_range, new_azimuth, np.arange(sar_data.shape[2])), axis=-1)

# Interpolate both datasets onto the new grid
new_sar_data = sar_interp(coords)
new_se_sar_data = se_sar_interp(coords)

# Merge the datasets
merged_data = np.maximum(new_sar_data, new_se_sar_data)

# Where one dataset has NaN and the other doesn't, use the non-NaN value
sar_nan = np.isnan(new_sar_data)
se_sar_nan = np.isnan(new_se_sar_data)
merged_data[sar_nan & ~se_sar_nan] = new_se_sar_data[sar_nan & ~se_sar_nan]
merged_data[~sar_nan & se_sar_nan] = new_sar_data[~sar_nan & se_sar_nan]

: 

In [24]:
import numpy as np

shape=(500, 500, 84)
gt_shape=(500, 500)

# Generate dummy input data
input_data = np.random.rand(*shape).astype(np.float32)

# Generate dummy ground truth data
ground_truth = np.random.rand(*(1274, 12953)).astype(np.float32)




In [26]:
ground_truth.ndim

2

In [28]:
# load cov chunks 
path = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_0_0_0.h5' 

# load h5 data 


In [29]:
import h5py
import numpy as np

def read_h5_file(filepath):
    """
    Read data from an HDF5 file.

    Args:
        filepath (str): Path to the HDF5 file.

    Returns:
        dict: A dictionary containing the datasets found in the file.
    """
    data = {}
    
    with h5py.File(filepath, 'r') as f:
        # Iterate through all datasets in the file
        for key in f.keys():
            # Read the dataset into a numpy array
            data[key] = np.array(f[key])
    
    return data

# Example usage

file_data = read_h5_file(path)

In [31]:
file_data = read_h5_file(path)
file_data.keys()
# dict_keys(['data', 'ground_truth'])

dict_keys(['data', 'ground_truth'])

In [51]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader



class ChunkDataset(Dataset):
    def __init__(self, chunk_files, patch_size=(32, 32, 84), stride=(16, 16)):
        self.chunk_files = chunk_files
        self.patch_size = patch_size
        self.stride = stride
        self.patches = self._create_patch_index()

    def _create_patch_index(self):
        patches = []
        for file_idx, file_path in enumerate(self.chunk_files):
            file_data = read_h5_file(file_path)
            data_shape = file_data['data'].shape
            
            for y in range(0, data_shape[0] - self.patch_size[0] + 1, self.stride[0]):
                for x in range(0, data_shape[1] - self.patch_size[1] + 1, self.stride[1]):
                    patches.append((file_idx, y, x))
        return patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        file_idx, y, x = self.patches[idx]
        file_path = self.chunk_files[file_idx]
        
        file_data = read_h5_file(file_path)
        
        data_patch = file_data['data'][y:y+self.patch_size[0], 
                                       x:x+self.patch_size[1], 
                                       :self.patch_size[2]]
        
        ground_truth_patch = file_data['ground_truth'][y:y+self.patch_size[0], 
                                                       x:x+self.patch_size[1]]
        
        return torch.from_numpy(data_patch), torch.from_numpy(ground_truth_patch)

# The rest of the code remains the same

# Usage
directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_300_covariance_matrix'
chunk_files = get_chunk_files(directory)

print(f"Found {len(chunk_files)} chunk files.")

# Specify which quadrants to use for each split
train_quadrants = [0, 1]  # Use quadrants 0 and 1 for training
val_quadrant = 2  # Use quadrant 2 for validation
test_quadrant = 3  # Use quadrant 3 for testing

train_files, val_files, test_files = split_chunks_by_quadrants(chunk_files, train_quadrants, val_quadrant, test_quadrant)

print(f"Train files: {len(train_files)}")
print(f"Val files: {len(val_files)}")
print(f"Test files: {len(test_files)}")

try:
    train_dataset = ChunkDataset(train_files, patch_size=(32, 32, 84), stride=(16, 16))
    print(f"Successfully created dataset with {len(train_dataset)} patches")
    
    train_loader, val_loader, test_loader = create_dataloaders(train_files, val_files, test_files, 
                                                               patch_size=(32, 32, 84), stride=(16, 16), 
                                                               batch_size=32)
    
    # Now you can use these loaders in your training loop
    for batch_data, batch_ground_truths in train_loader:
        print(f"Batch data shape: {batch_data.shape}")
        print(f"Batch ground truth shape: {batch_ground_truths.shape}")
        break
except Exception as e:
    print(f"An error occurred: {str(e)}")
    
    # Print contents of the first file for debugging
    if chunk_files:
        print(f"\nContents of the first file ({chunk_files[0]}):")
        file_data = read_h5_file(chunk_files[0])
        print(f"Keys: {list(file_data.keys())}")
        for key in file_data.keys():
            print(f"'{key}' shape: {file_data[key].shape}")

Found 36 chunk files.
Train files: 18
Val files: 9
Test files: 9
Successfully created dataset with 72 patches
Batch data shape: torch.Size([32, 32, 32, 84])
Batch ground truth shape: torch.Size([32, 32, 32])


In [ ]:
/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_0_0.h5

In [88]:
def reconstruct_cube_from_chunks(chunk_directory, original_shape, chunk_sizes):
    """
    Reconstruct a cube from chunks of various sizes.
    
    :param chunk_directory: Path to the directory containing chunk files
    :param original_shape: Original shape of the full cube (x, y, z)
    :param chunk_sizes: Tuple of (x_chunk_size, y_chunk_size, z_chunk_size)
    :return: Reconstructed numpy array
    """
    chunk_files = sorted(glob.glob(chunk_directory + '*_*_*.h5'))
    if len(chunk_files) == 0:
        raise ValueError('Chunks list empty')
    
    # Initialize an empty array to hold the reconstructed data
    reconstructed_cube = np.zeros(original_shape, dtype=np.float32)
    
    max_x_idx, max_y_idx, max_z_idx = -1, -1, -1
    
    for chunk_file in chunk_files:
        print('Processing chunk file:', chunk_file)
        # Extract chunk indices from filename
        filename = os.path.basename(chunk_file)
        x_idx, y_idx, z_idx = map(int, filename.split('.')[0].split('_')[-3:])
        
        max_x_idx = max(max_x_idx, x_idx)
        max_y_idx = max(max_y_idx, y_idx)
        max_z_idx = max(max_z_idx, z_idx)
        print(max_x_idx, max_y_idx, max_z_idx)
        
        # Load the chunk data
        with h5py.File(chunk_file, 'r') as f:
            if 'data' not in f:
                raise KeyError(f"'data' key not found in file {chunk_file}")
            chunk_data = f['data'][:]
        
        print('Chunk shape:', chunk_data.shape)
        
        # Calculate the position of this chunk in the final array
        x_start = x_idx * chunk_sizes[0]
        y_start = y_idx * chunk_sizes[1]
        z_start = z_idx * chunk_sizes[2]
        
        # Calculate end positions
        x_end = min(x_start + chunk_sizes[0], original_shape[0])
        y_end = min(y_start + chunk_sizes[1], original_shape[1])
        z_end = min(z_start + chunk_sizes[2], original_shape[2])
        
        # Check if the chunk fits within the original shape
        if x_end > original_shape[0] or y_end > original_shape[1] or z_end > original_shape[2]:
            raise ValueError(f"Chunk at position ({x_idx}, {y_idx}, {z_idx}) exceeds the original shape. "
                             f"Chunk ends at ({x_end}, {y_end}, {z_end}), but original shape is {original_shape}")
        
        # Place the chunk in the correct position
        reconstructed_cube[x_start:x_end, y_start:y_end, z_start:z_end] = chunk_data[:x_end-x_start, :y_end-y_start, :z_end-z_start]
    
    # Check if we've filled the entire cube
    expected_x_chunks = (original_shape[0] + chunk_sizes[0] - 1) // chunk_sizes[0]
    expected_y_chunks = (original_shape[1] + chunk_sizes[1] - 1) // chunk_sizes[1]
    expected_z_chunks = (original_shape[2] + chunk_sizes[2] - 1) // chunk_sizes[2]
    if max_x_idx + 1 < expected_x_chunks or max_y_idx + 1 < expected_y_chunks or max_z_idx + 1 < expected_z_chunks:
        raise ValueError(f"Mismatch in number of chunks. Expected {expected_x_chunks}x{expected_y_chunks}x{expected_z_chunks}, "
                         f"but got {max_x_idx + 1}x{max_y_idx + 1}x{max_z_idx + 1}")
    
    # Check if there are any unfilled areas
    if np.any(reconstructed_cube == 0):
        raise ValueError("The reconstructed cube contains unfilled areas (zeros)")
    
    return reconstructed_cube


chunk_directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/'

original_shape = (200, 350, 84)  # Example shape
chunk_sizes = (200, 200, 84)  # Example chunk sizes
reconstructed_cube = reconstruct_cube_from_chunks(chunk_directory, original_shape, chunk_sizes)


Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_0_0.h5
0 0 0
Chunk shape: (200, 200, 84)
Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_1_0.h5
0 1 0
Chunk shape: (200, 150, 84)


In [91]:

chunk_directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_1000_covariance_matrix/computed_chunks/'

original_shape = (1000, 1000, 84)  # Example shape
chunk_sizes = (500, 500, 84)  # Example chunk sizes
reconstructed_cube = reconstruct_cube_from_chunks(chunk_directory, original_shape, chunk_sizes)


Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_1000_covariance_matrix/computed_chunks/chunk_0_0_0.h5
0 0 0
Chunk shape: (500, 500, 84)
Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_1000_covariance_matrix/computed_chunks/chunk_0_1_0.h5
0 1 0
Chunk shape: (500, 500, 84)
Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_1000_covariance_matrix/computed_chunks/chunk_1_0_0.h5
1 1 0
Chunk shape: (500, 500, 84)
Processing chunk file: /home/grace.bcolverd/2024-ESL-BiomassSAR/size_1000_covariance_matrix/computed_chunks/chunk_1_1_0.h5
1 1 0
Chunk shape: (500, 500, 84)


In [92]:
reconstructed_cube

array([[[ 1.0000000e+00,  1.0000000e+00,  1.0000000e+00, ...,
          3.0727398e-02, -4.9219497e-02,  6.1860871e-02],
        [ 1.0000000e+00,  1.0000000e+00,  1.0000000e+00, ...,
          1.4965786e-02, -6.3341424e-02,  7.8716233e-02],
        [ 1.0000000e+00,  1.0000001e+00,  9.9999994e-01, ...,
          4.4343695e-02, -5.2890029e-02,  1.0352610e-01],
        ...,
        [ 1.0000000e+00,  9.9999988e-01,  9.9999994e-01, ...,
         -6.8506703e-02,  2.3523295e-02, -6.0646269e-02],
        [ 1.0000000e+00,  9.9999994e-01,  1.0000001e+00, ...,
         -6.4407825e-02,  2.7579946e-02, -6.8559878e-02],
        [ 1.0000000e+00,  9.9999994e-01,  1.0000000e+00, ...,
         -3.8506694e-02,  5.4109167e-02, -7.1953677e-02]],

       [[ 1.0000000e+00,  1.0000000e+00,  9.9999994e-01, ...,
          5.6698985e-02, -4.7799781e-02,  5.7686377e-02],
        [ 9.9999994e-01,  9.9999994e-01,  9.9999994e-01, ...,
          4.1179910e-02, -6.5511785e-02,  7.0343234e-02],
        [ 9.9999994e-01, 

In [83]:
# import glob 
# import dask.array as da
# import numpy as np
# from pathlib import Path
# import h5py

# def reconstruct_cube_from_chunks(chunk_directory, original_shape):
#     """
#     Reconstruct a cube from Dask chunks.
    
#     :param chunk_directory: Path to the directory containing chunk files
#     :param original_shape: Original shape of the full cube (x, y, z)
#     :return: Reconstructed numpy array
#     """
#     chunk_files = sorted(glob.glob(chunk_directory +'*_*_*.h5'))
#     if len(chunk_files)==0:
#         raise ValueError('Chunks list empty')
    
#     # Initialize an empty array to hold the reconstructed data
#     reconstructed_cube = np.zeros(original_shape, dtype=np.float32)
    
#     for chunk_file in chunk_files:
#         # Extract chunk indices from filename
#         print('chunk file ', chunk_file )
#         filename = os.path.basename(chunk_file)
#         x_idx, y_idx, z_idx = map(int, filename.split('.')[0].split('_')[-3:])
        
        
#         # Load the chunk data
#         with h5py.File(chunk_file, 'r') as f:
#             chunk_data = f['data'][:]
#         print('chunk shape '  , chunk_data.shape)
#         # Calculate the position of this chunk in the final array
#         x_start = sum(original_shape[0] // ((original_shape[0] + 99) // 100) for _ in range(x_idx))
#         y_start = sum(original_shape[1] // ((original_shape[1] + 99) // 100) for _ in range(y_idx))
#         z_start = 0  # Since z is not chunked
        
#         # Place the chunk in the correct position
#         x_end = x_start + chunk_data.shape[0]
#         y_end = y_start + chunk_data.shape[1]
#         z_end = z_start + chunk_data.shape[2]
        
#         reconstructed_cube[x_start:x_end, y_start:y_end, z_start:z_end] = chunk_data
    
#     return reconstructed_cube

# # Usage
# chunk_directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/*/'
# original_shape = (200, 350, 84)  # Adjust based on your full cube shape (x, y, z)

# reconstructed_data = reconstruct_cube_from_chunks(chunk_directory, original_shape)
# print(f"Reconstructed cube shape: {reconstructed_data.shape}")

chunk file  /home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_0_0.h5
chunk shape  (200, 200, 84)
chunk file  /home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_1_0.h5
chunk shape  (200, 150, 84)
Reconstructed cube shape: (200, 350, 84)


In [84]:
reconstructed_data

array([[[ 1.0000001e+00,  1.0000000e+00,  1.0000001e+00, ...,
          6.2633276e-02, -2.2704070e-02,  5.3447881e-03],
        [ 9.9999994e-01,  1.0000000e+00,  1.0000001e+00, ...,
          5.7087269e-02,  1.8007414e-02,  3.7125621e-02],
        [ 1.0000000e+00,  9.9999994e-01,  1.0000000e+00, ...,
          5.3080618e-02,  2.9069401e-02,  3.2028414e-02],
        ...,
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]],

       [[ 1.0000000e+00,  1.0000000e+00,  9.9999994e-01, ...,
          3.4745470e-02, -2.4573714e-02,  8.6638061e-03],
        [ 1.0000000e+00,  1.0000000e+00,  9.9999994e-01, ...,
          3.5527840e-02,  2.2406727e-02,  3.5096951e-02],
        [ 9.9999994e-01, 

In [79]:
import glob 
chunk_directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/*/*.h5'
glob.glob(chunk_directory)

['/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_1_0.h5',
 '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_200_covariance_matrix/computed_chunks/chunk_0_0_0.h5']

In [61]:
file_data['data'].shape
patch_size=(32,32,84) 
stride =(16,16,84) 
chunk_files=train_files

# file_path='/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_1_0_0.h5'
patches = []
for file_idx, file_path in enumerate(chunk_files):
    print('index ' , file_idx)
    file_data = read_h5_file(file_path)
    data_shape = file_data['data'].shape
    print(data_shape)
    
    for y in range(0, data_shape[0] - patch_size[0] + 1, stride[0]):
        for x in range(0, data_shape[1] - patch_size[1] + 1, stride[1]):
            patches.append((file_idx, y, x))
    print(len(patches))




index  0
(50, 50, 84)
4
index  1
(50, 50, 84)
8
index  2
(50, 50, 84)
12
index  3
(50, 50, 84)
16
index  4
(50, 50, 84)
20
index  5
(50, 50, 84)
24
index  6
(50, 50, 84)
28
index  7
(50, 50, 84)
32
index  8
(50, 50, 84)
36
index  9
(50, 50, 84)
40
index  10
(50, 50, 84)
44
index  11
(50, 50, 84)
48
index  12
(50, 50, 84)
52
index  13
(50, 50, 84)
56
index  14
(50, 50, 84)
60
index  15
(50, 50, 84)
64
index  16
(50, 50, 84)
68
index  17
(50, 50, 84)
72


In [58]:
len(patches)

72

In [53]:
500/32 * 15.6



243.75

In [33]:
file_data['data'].shape

(50, 50, 84)

In [34]:
file_data['ground_truth'].shape

(50, 50)

In [47]:
import os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class ChunkDataset(Dataset):
    def __init__(self, chunk_files, patch_size=(32, 32, 32), stride=(16, 16, 16)):
        self.chunk_files = chunk_files
        self.patch_size = patch_size
        self.stride = stride
        self.patches = self._create_patch_index()

    def _create_patch_index(self):
        patches = []
        for file_idx, file_path in enumerate(self.chunk_files):
            with h5py.File(file_path, 'r') as f:
                array_shape = f['array'].shape
            
            for z in range(0, array_shape[0] - self.patch_size[0] + 1, self.stride[0]):
                for y in range(0, array_shape[1] - self.patch_size[1] + 1, self.stride[1]):
                    for x in range(0, array_shape[2] - self.patch_size[2] + 1, self.stride[2]):
                        patches.append((file_idx, z, y, x))
        return patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        file_idx, z, y, x = self.patches[idx]
        file_path = self.chunk_files[file_idx]
        
        with h5py.File(file_path, 'r') as f:
            array_patch = f['array'][z:z+self.patch_size[0], 
                                     y:y+self.patch_size[1], 
                                     x:x+self.patch_size[2]]
            ground_truth_patch = f['ground_truth'][z:z+self.patch_size[0], 
                                                   y:y+self.patch_size[1], 
                                                   x:x+self.patch_size[2]]
        
        return torch.from_numpy(array_patch), torch.from_numpy(ground_truth_patch)

def get_chunk_files(directory):
    chunk_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.h5'):
                chunk_files.append(os.path.join(root, file))
    return chunk_files

def split_chunks_by_quadrants(chunk_files, train_quadrants, val_quadrant, test_quadrant):
    # Extract x and y coordinates from filenames
    coordinates = [tuple(map(int, file.split('_')[-3:-1])) for file in chunk_files]
    
    # Find the maximum x and y coordinates
    max_x = max(coord[0] for coord in coordinates)
    max_y = max(coord[1] for coord in coordinates)
    
    # Define quadrant boundaries
    mid_x, mid_y = max_x // 2, max_y // 2
    quadrants = {
        0: lambda x, y: x <= mid_x and y <= mid_y,
        1: lambda x, y: x > mid_x and y <= mid_y,
        2: lambda x, y: x <= mid_x and y > mid_y,
        3: lambda x, y: x > mid_x and y > mid_y
    }
    
    # Split files based on quadrants
    train_files = [f for f, (x, y) in zip(chunk_files, coordinates) if any(quadrants[q](x, y) for q in train_quadrants)]
    val_files = [f for f, (x, y) in zip(chunk_files, coordinates) if quadrants[val_quadrant](x, y)]
    test_files = [f for f, (x, y) in zip(chunk_files, coordinates) if quadrants[test_quadrant](x, y)]
    
    return train_files, val_files, test_files

def create_dataloaders(train_files, val_files, test_files, patch_size=(32, 32, 32), stride=(16, 16, 16), batch_size=32):
    train_dataset = ChunkDataset(train_files, patch_size, stride)
    val_dataset = ChunkDataset(val_files, patch_size, stride)
    test_dataset = ChunkDataset(test_files, patch_size, stride)
    print('test ' , test_dataset)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader

# Usage
directory = '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_300_covariance_matrix'
chunk_files = get_chunk_files(directory)

# Specify which quadrants to use for each split
train_quadrants = [0, 1]  # Use quadrants 0 and 1 for training
val_quadrant = 2  # Use quadrant 2 for validation
test_quadrant = 3  # Use quadrant 3 for testing

train_files, val_files, test_files = split_chunks_by_quadrants(chunk_files, train_quadrants, val_quadrant, test_quadrant)
print(len(train_files), len(val_files), len(test_files) ) 

train_loader, val_loader, test_loader = create_dataloaders(train_files, val_files, test_files, 
                                                           patch_size=(32, 32, 32), stride=(16, 16, 16))

# Now you can use these loaders in your training loop
for batch_arrays, batch_ground_truths in train_loader:
    # Your training code here
    pass

18 9 9


KeyError: "Unable to open object (object 'array' doesn't exist)"

In [48]:

ChunkDataset(train_files )

KeyError: "Unable to open object (object 'array' doesn't exist)"

In [36]:
def get_chunk_files(directory):
    chunk_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.h5'):
                chunk_files.append(os.path.join(root, file))
    return chunk_files


get_chunk_files('/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix')

['/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_0_1_0.h5',
 '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_1_1_0.h5',
 '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_0_0_0.h5',
 '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_1_0_0.h5']

In [37]:
 coordinates = [tuple(map(int, file.split('_')[-3:-1])) for file in chunk_files]
    
# Find the maximum x and y coordinates
max_x = max(coord[0] for coord in coordinates)
max_y = max(coord[1] for coord in coordinates)

# Define quadrant boundaries
mid_x, mid_y = max_x // 2, max_y // 2
quadrants = {
    0: lambda x, y: x <= mid_x and y <= mid_y,
    1: lambda x, y: x > mid_x and y <= mid_y,
    2: lambda x, y: x <= mid_x and y > mid_y,
    3: lambda x, y: x > mid_x and y > mid_y
}

# Split files based on quadrants
train_files = [f for f, (x, y) in zip(chunk_files, coordinates) if any(quadrants[q](x, y) for q in train_quadrants)]
val_files = [f for f, (x, y) in zip(chunk_files, coordinates) if quadrants[val_quadrant](x, y)]
test_files = [f for f, (x, y) in zip(chunk_files, coordinates) if quadrants[test_quadrant](x, y)]


In [38]:
train_files

['/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_0_0_0.h5',
 '/home/grace.bcolverd/2024-ESL-BiomassSAR/size_100_covariance_matrix/computed_chunks/chunk_1_0_0.h5']

In [39]:
train_dataset = ChunkDataset(train_files)

In [42]:
batch_size=4
train_dataset = ChunkDataset(train_files)
val_dataset = ChunkDataset(val_files)
test_dataset = ChunkDataset(test_files)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)